In [1]:
import pandas as pd

C:\Users\sonya\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\sonya\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [46]:
all_results_df = pd.read_csv("all_results_full.csv")
all_data_df = pd.read_csv("all_data_metrics.csv")
all_model_df = pd.read_csv("all_model_metrics.csv")

In [40]:
df.columns

Index(['sampler', 'model', 'n_train_before', 'n_train_after', 'ir_before',
       'ir_after', 'n3_before', 'n3_after', 'f1_fisher_mean_before',
       'f1_fisher_mean_after', 'f1_fisher_max_before', 'f1_fisher_max_after',
       'f1_fisher_min_before', 'f1_fisher_min_after', 'f2_overlap_mean_before',
       'f2_overlap_mean_after', 'f2_overlap_max_before',
       'f2_overlap_max_after', 'f2_overlap_min_before', 'f2_overlap_min_after',
       'status', 'error', 'balanced_accuracy', 'f1_macro', 'gmean_macro',
       'auc_roc_ovr_macro', 'dataset_id', 'dataset_name', 'group', 'ir_level',
       'overlap_level', 'cluster_level'],
      dtype='str')

In [47]:
results_ok = all_results_df[all_results_df["status"] == "ok"].copy()
data_ok = all_data_df[all_data_df["status"] == "ok"].copy()
model_ok = all_model_df[all_model_df["status"] == "ok"].copy()

## Базовый анализ: кто лучший в среднем

In [49]:
sampler_summary = (
    results_ok.groupby("sampler")[["balanced_accuracy", "f1_macro", "gmean_macro", "auc_roc_ovr_macro"]]
    .mean()
    .round(3)
    .sort_values("f1_macro", ascending=False)
)

print(sampler_summary)

                 balanced_accuracy  f1_macro  gmean_macro  auc_roc_ovr_macro
sampler                                                                     
KMeansSMOTE                  0.751     0.754        0.830              0.905
ADASYN                       0.720     0.705        0.808              0.897
SMOTE                        0.715     0.705        0.805              0.895
SVMSMOTE                     0.713     0.703        0.803              0.893
BorderlineSMOTE              0.715     0.701        0.804              0.892
NoSampler                    0.669     0.681        0.774              0.895
cluster_SMOTE                0.666     0.673        0.773              0.894
MWMOTE                       0.666     0.672        0.773              0.893
distance_SMOTE               0.665     0.672        0.772              0.893
CBSO                         0.666     0.671        0.773              0.892
DBSMOTE                      0.663     0.671        0.771              0.894

In [50]:
model_summary = (
    results_ok.groupby("model")[["balanced_accuracy", "f1_macro", "gmean_macro", "auc_roc_ovr_macro"]]
    .mean()
    .round(3)
    .sort_values("f1_macro", ascending=False)
)

print(model_summary)

                    balanced_accuracy  f1_macro  gmean_macro  \
model                                                          
RandomForest                    0.737     0.741        0.825   
LogisticRegression              0.596     0.578        0.719   

                    auc_roc_ovr_macro  
model                                  
RandomForest                    0.946  
LogisticRegression              0.843  


Вывод: 

1) Лучшим методом балансировки в среднем оказался KMeansSMOTE.

Он показал наилучшие результаты по всем основным метрикам, а значит, в рамках усреднённого анализа его можно считать наиболее эффективным среди рассмотренных sampler’ов.

2) Сильную группу методов составляют ADASYN, SMOTE, SVMSMOTE и BorderlineSMOTE.

Они уступают KMeansSMOTE, но тоже дают высокое среднее качество и заметно превосходят baseline без балансировки.

3) RandomForest в среднем существенно лучше LogisticRegression.

Это означает, что влияние методов балансировки лучше раскрывается на более гибкой и нелинейной модели. Для линейной модели улучшения выражены слабее.

## Сравнение с baseline

In [51]:
baseline_df = results_ok[results_ok["sampler"] == "NoSampler"].copy()
baseline_metrics = baseline_df[
    ["dataset_name", "model", "f1_macro", "gmean_macro", "auc_roc_ovr_macro"]
].rename(columns={
    "f1_macro": "f1_baseline",
    "gmean_macro": "gmean_baseline",
    "auc_roc_ovr_macro": "auc_baseline"
})

df_compare = results_ok.merge(
    baseline_metrics,
    on=["dataset_name", "model"],
    how="left"
)
df_compare["delta_f1"] = df_compare["f1_macro"] - df_compare["f1_baseline"]
df_compare["delta_gmean"] = df_compare["gmean_macro"] - df_compare["gmean_baseline"]
df_compare["delta_auc"] = df_compare["auc_roc_ovr_macro"] - df_compare["auc_baseline"]

quality_vs_baseline = (
    df_compare[df_compare["sampler"] != "NoSampler"]
    .groupby("sampler")[["delta_f1", "delta_gmean", "delta_auc"]]
    .mean()
    .round(3)
    .sort_values("delta_f1", ascending=False)
)

print(quality_vs_baseline)

                 delta_f1  delta_gmean  delta_auc
sampler                                          
SMOTE               0.023        0.030     -0.000
SVMSMOTE            0.022        0.029     -0.002
BorderlineSMOTE     0.020        0.030     -0.003
ADASYN              0.016        0.028     -0.002
KMeansSMOTE         0.007        0.009     -0.006
cluster_SMOTE      -0.008       -0.001     -0.001
MWMOTE             -0.009       -0.001     -0.002
distance_SMOTE     -0.009       -0.002     -0.002
CBSO               -0.010       -0.001     -0.003
DBSMOTE            -0.010       -0.003     -0.001
AHC                -0.322       -0.166        NaN


Сравнение результатов с baseline показало, что наибольший положительный эффект на качество классификации обеспечили методы SMOTE, SVMSMOTE, BorderlineSMOTE и ADASYN. Для этих методов наблюдался положительный прирост как F1-macro, так и G-mean, что свидетельствует об улучшении качества классификации по сравнению с обучением на исходных несбалансированных данных.

Метод KMeansSMOTE также продемонстрировал положительный эффект, однако он был заметно слабее по сравнению с классическими вариантами SMOTE. Методы cluster_SMOTE, MWMOTE, distance_SMOTE, CBSO и DBSMOTE не показали выигрыша относительно baseline и в среднем приводили к незначительному ухудшению результатов. Наиболее неудачным оказался метод AHC, который существенно снижал качество классификации.

## Анализ по моделям отдельно

In [52]:
lr_table = (
    df_compare[
        (df_compare["sampler"] != "NoSampler") &
        (df_compare["model"] == "LogisticRegression")
    ]
    .groupby("sampler")[["delta_f1", "delta_gmean", "delta_auc"]]
    .mean()
    .round(3)
    .sort_values("delta_f1", ascending=False)
)

rf_table = (
    df_compare[
        (df_compare["sampler"] != "NoSampler") &
        (df_compare["model"] == "RandomForest")
    ]
    .groupby("sampler")[["delta_f1", "delta_gmean", "delta_auc"]]
    .mean()
    .round(3)
    .sort_values("delta_f1", ascending=False)
)

print("LogisticRegression")
print(lr_table)

print("\nRandomForest")
print(rf_table)

LogisticRegression
                 delta_f1  delta_gmean  delta_auc
sampler                                          
KMeansSMOTE         0.014        0.014     -0.008
SMOTE               0.001        0.022     -0.004
SVMSMOTE           -0.005        0.017     -0.007
distance_SMOTE     -0.009       -0.001     -0.002
CBSO               -0.010       -0.001     -0.003
MWMOTE             -0.011       -0.002     -0.002
cluster_SMOTE      -0.011       -0.002     -0.002
DBSMOTE            -0.015       -0.004     -0.002
BorderlineSMOTE    -0.016        0.014     -0.010
ADASYN             -0.017        0.014     -0.007
AHC                -0.266       -0.134        NaN

RandomForest
                 delta_f1  delta_gmean  delta_auc
sampler                                          
BorderlineSMOTE     0.055        0.046      0.003
ADASYN              0.050        0.043      0.003
SVMSMOTE            0.048        0.041      0.003
SMOTE               0.046        0.038      0.003
KMeansSMOTE      

Анализ результатов по моделям показал, что влияние методов балансировки существенно зависит от используемого классификатора. Для LogisticRegression применение oversampling в большинстве случаев не приводило к улучшению F1-macro: положительный эффект наблюдался только для KMeansSMOTE и в незначительной степени для SMOTE. В то же время для RandomForest методы BorderlineSMOTE, ADASYN, SVMSMOTE и SMOTE обеспечили заметный прирост как F1-macro, так и G-mean. Это свидетельствует о том, что нелинейная ансамблевая модель значительно лучше использует изменения структуры данных, возникающие после применения методов oversampling.

## Анализ влияния сложности данных

In [53]:
by_ir = (
    df_compare[df_compare["sampler"] != "NoSampler"]
    .groupby(["ir_level", "sampler"])[["delta_f1", "delta_gmean", "delta_auc"]]
    .mean()
    .round(3)
    .sort_values(["ir_level", "delta_f1"], ascending=[True, False])
)

print(by_ir)

                           delta_f1  delta_gmean  delta_auc
ir_level  sampler                                          
high_ir   BorderlineSMOTE     0.038        0.051     -0.004
          SMOTE               0.038        0.048     -0.001
          SVMSMOTE            0.038        0.045     -0.002
          ADASYN              0.030        0.046     -0.003
          KMeansSMOTE        -0.006        0.004      0.000
          cluster_SMOTE      -0.012        0.001     -0.002
          distance_SMOTE     -0.015       -0.002     -0.004
          CBSO               -0.016        0.001     -0.005
          DBSMOTE            -0.016       -0.004     -0.002
          MWMOTE             -0.017       -0.001     -0.004
          AHC                -0.258       -0.130        NaN
low_ir    SMOTE               0.011        0.014     -0.001
          SVMSMOTE            0.011        0.015     -0.001
          KMeansSMOTE         0.007        0.008     -0.005
          BorderlineSMOTE     0.006     

Анализ результатов по уровням дисбаланса показал, что эффективность методов oversampling существенно зависит от величины IR. Наибольший положительный эффект наблюдается при высоком уровне дисбаланса, где методы SMOTE, SVMSMOTE, BorderlineSMOTE и ADASYN обеспечивают наиболее выраженный прирост F1-macro и G-mean относительно baseline. При среднем уровне дисбаланса положительный эффект сохраняется, однако становится менее выраженным. При низком уровне дисбаланса выигрыш от применения sampler’ов минимален, а в ряде случаев методы балансировки оказываются практически бесполезными. Это позволяет сделать вывод, что необходимость oversampling и выбор конкретного метода во многом определяются исходным уровнем дисбаланса классов.

In [54]:
by_overlap = (
    df_compare[df_compare["sampler"] != "NoSampler"]
    .groupby(["overlap_level", "sampler"])[["delta_f1", "delta_gmean", "delta_auc"]]
    .mean()
    .round(3)
    .sort_values(["overlap_level", "delta_f1"], ascending=[True, False])
)

print(by_overlap)

                                delta_f1  delta_gmean  delta_auc
overlap_level  sampler                                          
high_overlap   ADASYN              0.040        0.050     -0.001
               BorderlineSMOTE     0.040        0.046     -0.004
               SMOTE               0.039        0.044     -0.001
               SVMSMOTE            0.038        0.039     -0.004
               KMeansSMOTE         0.011        0.016     -0.012
               MWMOTE             -0.010        0.002     -0.002
               distance_SMOTE     -0.010        0.000     -0.003
               cluster_SMOTE      -0.011        0.000     -0.000
               CBSO               -0.012        0.002     -0.004
               DBSMOTE            -0.013       -0.002     -0.000
               AHC                -0.221       -0.105        NaN
low_overlap    SMOTE               0.008        0.018     -0.000
               SVMSMOTE            0.005        0.018     -0.001
               KMeansSMOT

Анализ результатов по уровням overlap показал, что эффективность методов oversampling существенно зависит от степени перекрытия классов. Наиболее выраженный положительный эффект наблюдается при высоком overlap, где методы ADASYN, BorderlineSMOTE, SMOTE и SVMSMOTE обеспечивают наибольший прирост F1-macro и G-mean относительно baseline. При среднем overlap те же методы сохраняют положительный эффект, однако он становится менее выраженным. При низком overlap выигрыш от балансировки минимален, а некоторые методы, ориентированные на сложные пограничные области, могут даже ухудшать качество классификации. Это позволяет сделать вывод, что overlap является важным критерием выбора метода oversampling.

In [55]:
by_clusters = (
    df_compare[df_compare["sampler"] != "NoSampler"]
    .groupby(["cluster_level", "sampler"])[["delta_f1", "delta_gmean", "delta_auc"]]
    .mean()
    .round(3)
    .sort_values(["cluster_level", "delta_f1"], ascending=[True, False])
)

print(by_clusters)

                                 delta_f1  delta_gmean  delta_auc
cluster_level   sampler                                          
high_clusters   SVMSMOTE            0.027        0.032     -0.003
                SMOTE               0.026        0.034     -0.002
                BorderlineSMOTE     0.024        0.033     -0.006
                KMeansSMOTE         0.020        0.019     -0.009
                ADASYN              0.019        0.031     -0.003
                CBSO               -0.013       -0.002     -0.004
                DBSMOTE            -0.013       -0.004     -0.001
                MWMOTE             -0.014       -0.003     -0.002
                cluster_SMOTE      -0.014       -0.003     -0.000
                distance_SMOTE     -0.015       -0.005     -0.003
                AHC                -0.239       -0.117        NaN
low_clusters    SMOTE               0.008        0.019     -0.001
                SVMSMOTE            0.003        0.017     -0.001
          

Анализ результатов по уровню кластерной сложности показал, что увеличение числа кластеров внутри классов в целом повышает полезность методов oversampling. Наиболее выраженный положительный эффект наблюдается при среднем и высоком числе кластеров, где методы BorderlineSMOTE, ADASYN, SMOTE и SVMSMOTE обеспечивают заметный прирост F1-macro и G-mean относительно baseline. При низком числе кластеров выигрыш от балансировки существенно слабее, что указывает на меньшую необходимость применения sampler’ов при более простой внутренней структуре классов. Метод KMeansSMOTE демонстрирует улучшение результатов при высокой кластерной сложности, однако в рассматриваемом эксперименте уступает классическим вариантам SMOTE.

## Комбинированный анализ: для каких условий какой метод лучший

In [56]:
best_by_conditions = (
    df_compare[df_compare["sampler"] != "NoSampler"]
    .groupby(["ir_level", "overlap_level", "cluster_level", "sampler"])["delta_f1"]
    .mean()
    .reset_index()
)

best_sampler_per_case = (
    best_by_conditions.sort_values("delta_f1", ascending=False)
    .groupby(["ir_level", "overlap_level", "cluster_level"])
    .first()
    .reset_index()
)

print(best_sampler_per_case)

     ir_level   overlap_level    cluster_level          sampler  delta_f1
0     high_ir    high_overlap    high_clusters  BorderlineSMOTE  0.083222
1     high_ir    high_overlap     low_clusters            SMOTE  0.035553
2     high_ir    high_overlap  medium_clusters  BorderlineSMOTE  0.099159
3     high_ir     low_overlap    high_clusters            SMOTE  0.063218
4     high_ir     low_overlap     low_clusters   distance_SMOTE  0.004794
5     high_ir     low_overlap  medium_clusters            SMOTE  0.025726
6     high_ir  medium_overlap    high_clusters         SVMSMOTE  0.033876
7     high_ir  medium_overlap     low_clusters         SVMSMOTE  0.008442
8     high_ir  medium_overlap  medium_clusters  BorderlineSMOTE  0.090554
9      low_ir    high_overlap    high_clusters            SMOTE  0.051238
10     low_ir    high_overlap     low_clusters             CBSO  0.005011
11     low_ir    high_overlap  medium_clusters            SMOTE  0.017041
12     low_ir     low_overlap    high_

Комбинированный анализ показал, что выбор метода oversampling должен учитывать одновременно уровень дисбаланса, степень overlap и кластерную структуру данных. Универсальным и наиболее устойчивым методом в большинстве сценариев оказался SMOTE, который чаще других демонстрировал наилучший прирост F1-macro. Для более сложных сценариев, характеризующихся высоким уровнем overlap и/или дисбаланса, эффективными также оказывались BorderlineSMOTE, SVMSMOTE и ADASYN. В частности, BorderlineSMOTE демонстрировал высокую эффективность на данных с сочетанием высокого дисбаланса и сильного перекрытия классов, тогда как ADASYN чаще выигрывал при среднем дисбалансе и высоком overlap. Методы KMeansSMOTE, CBSO, distance_SMOTE и MWMOTE показывали преимущество лишь в отдельных частных сценариях и не могут рассматриваться как универсальные рекомендации.